# Upscale — the model shaped for the browser

`upscale_nn.ipynb` asked whether a learned upscaler beats the static filters. This one
asks the question the client actually has: **how fast can a learned upscaler be in a
browser**, and what does each way of making it faster cost in quality.

Same data, same metric, same five phases. What is different is that every design decision
here is made against a measurement taken *in the browser*, on the benchmark page, through
the runtime that will run it — because every one of the tricks below is invisible to
torch, and two of them measure backwards on a desktop CPU.

The four that paid, in the order they pay:

1. **No `Resize`.** The bicubic skip of `upscale_nn` is an interpolation, and an
   interpolation by an integer factor is a convolution — one that can be *written down*
   as weights. Replacing the op with a 5×5 `Conv` is bit-identical arithmetic (2.4e-07
   apart, which is float32 noise) and it takes the model off the one operator the
   runtimes handle worst: **55.6 ms → 1.9 ms** per tile on the desktop CPU runtime, and
   the graph drops inside the WebGPU op budget.
2. **Add at low resolution.** `DepthToSpace` is linear, so `DTS(a) + DTS(b)` is
   `DTS(a + b)`. Both branches carry `scale²` channels, so the residual add moves before
   the shuffle: a quarter of the elements touched, and one shuffle in the graph instead
   of two.
3. **Narrower, at a stated price.** The body is the whole cost; the ladder below prices
   each width in dB so the choice is a trade rather than a guess.
4. **float16.** Half the weights, half the bandwidth, the same PSNR to two decimals —
   and about 25% off the tile on a card with `shader-f16`.

Three more live outside the graph, in how the page runs it: **GPU-resident tensors**
(25× — the biggest single number in this notebook), a **reused output buffer** (17%), and
**graph capture** (another 17%).

And two that did not: **int8** costs 0.10 dB and is *130× slower* in the browser, because
its Q/DQ pairs fall off the WebGPU provider; **depthwise separable** convolutions lose
0.5 dB and save nothing measurable. Both are below, with the numbers, because a trick that
does not pay is worth the same as one that does if it stops the next person paying for it
twice.

The result: **0.19 ms per tile, 7 ms for a 1080p frame** on an Ampere card — inside the
60 fps budget with the decode still to pay for — at **+6.9 dB over bicubic** on this
dataset, against 18 ms and +7.8 dB for `upscale_nn`. At the tile size the sweep argues for
it is **5 ms a frame**, and on a desktop stream, where most tiles do not change between
frames, it is a fraction of that again.

In [ ]:
"""Configuration — every knob this notebook has."""

from pathlib import Path

DATA_DIR = Path("data/processed")
RAW_DIR = Path("data/raw")
CHECKPOINT_DIR = Path("checkpoints")
SAMPLE_DIR = Path("data/samples")

# The body: two convolutions at LR, then the tail that feeds the shuffle. The ladder in
# phase 2 is what these two numbers were chosen from.
WIDTHS = (32, 16)
STEM = 5                       # first kernel; the skip uses the same reach
SKIP = 5                       # a x2 bicubic filter needs 4 taps, so 5 is the smallest odd fit

EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 0

# Tiles. The page's shared step is 128, which is what every other model on it is measured
# at; 256 is what the browser sweep argues for, and both get published.
DEPLOY_STEP = 256

BENCH_RESOLUTIONS = [(480, 270), (640, 360), (960, 540), (1280, 720)]

In [ ]:
"""Imports, device, and the manifest the preprocessing notebook wrote."""

import json
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "manifest.json").exists():
    raise RuntimeError(f"no dataset in {DATA_DIR} — run data_preprocess.ipynb first")

manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
SCALE = manifest["scale"]

print("device      ", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("torch       ", torch.__version__)
print("scale       ", SCALE)
print("patch       ", manifest["patch_lr"], "->", manifest["patch_hr"])

## 1. Data

The same patch pairs as `upscale_nn.ipynb`, loaded the same way — memory-mapped `.npy`,
the eight-way flip/rotate augmentation on the training split only, `val` held out by
source frame. That notebook carries the reasoning; this one only needs the loaders.

In [ ]:
"""Patch pairs as float tensors in [0, 1], with the flip/rotate augmentation."""

class PatchPairs(Dataset):
    def __init__(self, directory, split, manifest, augment):
        info = manifest["splits"][split]
        self.split = split
        self.frames = info["frames"]
        self.lr = np.load(directory / info["lr"], mmap_mode="r")
        self.hr = np.load(directory / info["hr"], mmap_mode="r")
        self.augment = augment

    def __len__(self):
        return len(self.lr)

    def __getitem__(self, index):
        lr = np.asarray(self.lr[index])
        hr = np.asarray(self.hr[index])

        if self.augment:
            turns = int(torch.randint(0, 4, ()))
            lr, hr = np.rot90(lr, turns), np.rot90(hr, turns)
            if int(torch.randint(0, 2, ())):
                lr, hr = lr[:, ::-1], hr[:, ::-1]

        to_tensor = lambda a: torch.from_numpy(
            np.ascontiguousarray(a.transpose(2, 0, 1))).float().div_(255.0)
        return to_tensor(lr), to_tensor(hr)


train_set = PatchPairs(DATA_DIR, "train", manifest, augment=True)
val_set = PatchPairs(DATA_DIR, "val", manifest, augment=False)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

print(f"{'split':>6}{'patches':>9}{'frames':>8}{'batches':>9}  augment")
for dataset, loader in [(train_set, train_loader), (val_set, val_loader)]:
    print(f"{dataset.split:>6}{len(dataset):9}{len(dataset.frames):8}{len(loader):9}"
          f"  {'yes' if dataset.augment else 'no'}")

## 2. Model

### The skip is a convolution

`upscale_nn` predicts a residual on top of `F.interpolate(mode="bicubic")`, which the
exporter turns into `Resize`. That one node is the single most expensive thing in the
graph, everywhere it runs: **48 ms** per tile on the desktop CPU provider, **20.5 ms** on
WASM — against 0.6 ms for the same runtime's linear mode. It is not the four extra taps;
it is one unoptimised kernel, and the model was sitting on it.

It also does not have to be there. Bicubic ×2 is a *linear, shift-invariant* operator with
an integer stride, which is the definition of a convolution: each of the four output
phases is a fixed 4-tap filter of the input. So it can be written as a `Conv` producing
`3 · scale²` channels followed by the `DepthToSpace` that is already in the graph — and
the weights need not be derived from the cubic formula, because the operator itself will
hand them over one impulse at a time. The cell below builds them that way and asserts the
result matches `F.interpolate` to float32 noise.

Two things follow from a skip made of weights. It is **in the op budget** — `Conv`,
`Relu`, `Add`, `DepthToSpace` and nothing else — and it *could* be trained. Left free it
learns something slightly different and scores no better (31.46 dB against 31.62 dB over
100 epochs), so it stays frozen: a frozen bicubic skip means an untrained model is exactly
the bicubic baseline, and every dB the training loop prints is honest gain.

### The add moves to low resolution

`DepthToSpace` is linear, so shuffling both branches and adding is the same arithmetic as
adding and shuffling once — on a quarter of the elements, with one node instead of two.
Both branches already carry `3 · scale²` channels, so nothing has to be reshaped for it.

### What it costs

| body | val PSNR | vs bicubic | params | MACs / LR px | ORT CPU ms/tile |
| --- | --- | --- | --- | --- | --- |
| 64-32 (`upscale_nn`'s body) | 31.62 dB | +8.21 | 27,696 | 27,588 | 1.89 |
| 48-24 | 31.08 dB | +7.67 | 17,544 | 17,460 | 1.35 |
| 32-32-32 (deeper) | 31.47 dB | +8.06 | 25,296 | 25,188 | 1.73 |
| **32-16** | **30.43 dB** | **+7.02** | **9,696** | **9,636** | **1.07** |
| 24-12 | 29.51 dB | +6.10 | 6,636 | 6,588 | 0.93 |
| 16-8 | 28.53 dB | +5.12 | 4,152 | 4,116 | 0.80 |

All at 100 epochs on the same data, same seed, same recipe — and read them to ±0.15 dB:
this notebook's own run of the 32-16 row lands at 30.28 dB rather than 30.43, on a
different draw of the same recipe, which is the size of the noise 76 validation patches
support. Two shapes that looked
promising and were not: a **depthwise-separable** body loses 0.5 dB at the same cost
(29.39 dB at 10,756 MACs — the pointwise mixing it saves is the part that was working),
and a **strided stem** running the body at half resolution loses 1 dB (29.46 dB) for a
saving that a smaller width buys more cheaply.

**32-16 is the shipped shape**: a third of the MACs of the 64-32 body for 1.2 dB, and it
is the point where a 1080p frame fits the 60 fps budget in the browser with room left for
the decode. The wider body is one constant away for anyone who would rather spend it.

In [ ]:
"""The bicubic filter, recovered from the operator it replaces."""

def bicubic(x, scale=None):
    return F.interpolate(x, scale_factor=scale or SCALE, mode="bicubic", align_corners=False)


def bicubic_kernel(scale=SCALE, size=SKIP):
    """The x2 bicubic upsampling filter as `scale²` kernels, one per output phase.

    Recovered rather than derived: the operator is linear, so pushing a unit impulse
    through `F.interpolate` at every position of a `size`x`size` window reads its weights
    straight out of it - no cubic formula, and no guess about which `a` torch uses.
    """
    impulse = torch.zeros(1, 1, size, size)
    weights = torch.zeros(scale * scale, size, size)
    centre = size // 2
    for y in range(size):
        for x in range(size):
            impulse.zero_()
            impulse[0, 0, y, x] = 1.0
            out = bicubic(impulse, scale)[0, 0]
            for dy in range(scale):
                for dx in range(scale):
                    weights[dy * scale + dx, y, x] = out[centre * scale + dy, centre * scale + dx]
    return weights


class BicubicSkip(nn.Module):
    """Bicubic x2 as one convolution, so the graph carries no Resize."""

    def __init__(self, scale=SCALE, size=SKIP, trainable=False):
        super().__init__()
        self.conv = nn.Conv2d(3, 3 * scale * scale, size, padding=size // 2, bias=False)
        phases = bicubic_kernel(scale, size)
        weight = torch.zeros_like(self.conv.weight)
        for channel in range(3):
            for phase in range(scale * scale):
                # Each colour keeps to itself; the cross terms stay zero.
                weight[channel * scale * scale + phase, channel] = phases[phase]
        with torch.no_grad():
            self.conv.weight.copy_(weight)
        self.conv.weight.requires_grad_(trainable)

    def forward(self, x):
        return self.conv(x)


# The claim is that this is the same operator, so it is checked rather than asserted in
# prose. Only the interior: at the border the conv pads with zeros and interpolate does
# not, which is the same difference tiled inference has at the frame edge.
probe = torch.rand(1, 3, 24, 24)
rebuilt = nn.PixelShuffle(SCALE)(BicubicSkip()(probe))
edge = 2 * SKIP
difference = (rebuilt[..., edge:-edge, edge:-edge]
              - bicubic(probe)[..., edge:-edge, edge:-edge]).abs().max().item()
assert difference < 1e-5, difference
print(f"conv skip vs F.interpolate: max abs difference {difference:.1e} in the interior")
print(f"({SKIP}x{SKIP} kernel, {3 * SCALE * SCALE} channels, "
      f"{3 * 3 * SCALE * SCALE * SKIP * SKIP} weights, frozen)")

In [ ]:
"""The model: a frozen bicubic skip, a small body, one add at LR, one shuffle."""

class WebUpscaler(nn.Module):
    def __init__(self, widths=WIDTHS, stem=STEM, skip=SKIP, scale=SCALE):
        super().__init__()
        self.scale = scale
        self.skip = BicubicSkip(scale, skip)

        layers = []
        width = 3
        for index, out in enumerate(widths):
            kernel = stem if index == 0 else 3
            layers += [nn.Conv2d(width, out, kernel, padding=kernel // 2),
                       nn.ReLU(inplace=True)]
            width = out
        self.body = nn.Sequential(*layers)
        self.tail = nn.Conv2d(width, 3 * scale * scale, 3, padding=1)
        self.shuffle = nn.PixelShuffle(scale)

        # Start as the identity on top of bicubic: epoch 0 measures exactly the baseline.
        nn.init.zeros_(self.tail.weight)
        nn.init.zeros_(self.tail.bias)

        # The halo the tiling needs: how far one output pixel reaches into the input.
        # stem//2 for the first convolution, one more per 3x3 after it, and the skip's
        # own reach, whichever is larger.
        self.halo = max(stem // 2 + len(widths), skip // 2)

    def forward(self, x):
        # DepthToSpace is linear, so the add happens at LR - a quarter of the elements -
        # and one shuffle serves both branches.
        return self.shuffle(self.skip(x) + self.tail(self.body(x)))


def psnr(prediction, target):
    mse = F.mse_loss(prediction.clamp(0, 1), target, reduction="none").mean(dim=(1, 2, 3))
    return (10.0 * torch.log10(1.0 / mse.clamp_min(1e-12))).mean().item()


model = WebUpscaler().to(DEVICE)
print(model, "\n")

print(f"{'parameter':<20}{'shape':<22}{'count':>9}  trained")
for name, parameter in model.named_parameters():
    print(f"{name:<20}{str(tuple(parameter.shape)):<22}{parameter.numel():>9,}"
          f"  {'yes' if parameter.requires_grad else 'no (bicubic)'}")

parameters = sum(p.numel() for p in model.parameters())
learned = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{'total':<20}{'':<22}{parameters:>9,}   ({learned:,} learned, "
      f"{parameters * 4 / 1e6:.2f} MB as float32)")
print(f"\nscale x{SCALE}, halo {model.halo} LR pixels")

In [ ]:
"""Multiply-accumulates per LR pixel, and the output sizes."""

def macs_per_lr_pixel(model, size=32):
    """Counted from the convolutions that actually run, not from the parameter count -
    a 5x5 kernel over one pixel is 25 MACs per input channel however few weights it has."""
    total = 0

    def hook(module, inputs, output):
        nonlocal total
        kernel = module.kernel_size[0] * module.kernel_size[1]
        pixels = output.shape[2] * output.shape[3]
        total += (module.in_channels // module.groups) * module.out_channels * kernel * pixels

    handles = [m.register_forward_hook(hook) for m in model.modules() if isinstance(m, nn.Conv2d)]
    with torch.no_grad():
        model(torch.zeros(1, 3, size, size, device=DEVICE))
    for handle in handles:
        handle.remove()
    return total / (size * size)


print(f"{macs_per_lr_pixel(model):,.0f} MACs per LR pixel "
      f"({macs_per_lr_pixel(model) * 960 * 540 / 1e9:.1f} GMAC for a 1080p frame)\n")

print(f"{'input':>20}  {'output':>20}")
for width, height in [(manifest["patch_lr"],) * 2] + BENCH_RESOLUTIONS:
    with torch.no_grad():
        shape = tuple(model(torch.zeros(1, 3, height, width, device=DEVICE)).shape)
    assert shape == (1, 3, height * SCALE, width * SCALE), shape
    print(f"{f'1x3x{height}x{width}':>20}  {'x'.join(str(v) for v in shape):>20}")

### What `Resize` was costing

The claim that started the redesign, measured here rather than quoted: the same weights,
the same arithmetic, exported twice — once with the skip as `F.interpolate` and once as
the convolution above — and timed through the runtime that has to run them.

In [ ]:
"""Export both skips and time them, so the reason for the redesign is in the output."""

import onnxruntime as ort

import webexport


class ResizeSkipUpscaler(WebUpscaler):
    """The same model with the skip left as an interpolation - `upscale_nn`'s shape."""

    def forward(self, x):
        return bicubic(x) + self.shuffle(self.tail(self.body(x)))


def time_graph(path, size, dtype=np.float32, seconds=0.5):
    """Milliseconds per tile through onnxruntime, on the input the graph was pinned to."""
    options = ort.SessionOptions()
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])
    name = session.get_inputs()[0].name
    sample = np.random.rand(1, 3, size, size).astype(dtype)
    for _ in range(3):
        session.run(None, {name: sample})

    started, runs = time.perf_counter(), 0
    while time.perf_counter() - started < seconds:
        session.run(None, {name: sample})
        runs += 1
    return (time.perf_counter() - started) / runs * 1000


size = webexport.TILE_STEP + 2 * model.halo
comparison = CHECKPOINT_DIR / "comparison"
comparison.mkdir(exist_ok=True)

variants = {"conv skip (this model)": model,
            "Resize skip (upscale_nn)": ResizeSkipUpscaler().to(DEVICE)}
for label, variant in variants.items():
    info = webexport.export(variant, f"skip_{'conv' if 'conv' in label else 'resize'}.onnx",
                            size, models_dir=comparison)
    ops = ", ".join(f"{op}x{count}" for op, count in sorted(info["ops"].items()))
    print(f"{label:<26} {time_graph(info['path'], size):7.2f} ms/tile   {ops}")
    if info["outside_budget"]:
        print(f"{'':<26} outside the WebGPU op budget: {', '.join(info['outside_budget'])}")

## 3. Training

Unchanged from `upscale_nn` — L1, Adam, cosine schedule, the validation split measured
after every epoch, best weights kept by that PSNR — with one number moved: **100 epochs
rather than 12**.

That is not a tuning preference, it is a correction. 268 patches at batch 32 is 8 steps
an epoch, so 12 epochs is 96 optimizer steps, and the reference model was nowhere near
converged when it was measured: the same architecture, same data, trained to 100 epochs,
goes from **+0.90 dB to +7.84 dB** over bicubic. Every quality number in `upscale_nn.ipynb`
and the summary that quotes it is a number about a model that had barely started.

Read the size of that jump with the dataset in mind. Twelve synthetic frames of drawn
windows and text, degraded by a clean bicubic downscale with nothing else in it, is a
narrow and very learnable distribution — inverting a known linear operator on repetitive
content is the easiest problem in the family, and a real capture with a codec in the
degradation will not give up 7 dB. What the number supports is *relative*: these are the
conditions under which the shapes below were compared with each other.

### One thing the conv skip does not inherit: the border

`Resize` extends the edge pixel when it runs out of input; a convolution pads with zeros.
So the two are the same operator in the interior and *not* at the outermost ring, and on a
64 px patch that ring is 12% of the pixels — enough that an untrained model measures 0.9 dB
below the bicubic it is supposed to start at.

That is the frame border, and it is the same border the halo already throws away: under
tiled inference every tile is fed real pixels around its step and the ring never reaches
the output. It matters at the true edge of a whole frame, where it is a two-pixel rim, and
training absorbs it anyway. So the starting point is checked where the claim actually holds
— in the interior — and every other number in this notebook is whole-patch, comparable with
the other two notebooks.

In [ ]:
"""Average PSNR over a loader, for the model and for the bicubic baseline."""

@torch.no_grad()
def evaluate(model, loader, border=0):
    """`border` drops that many HR pixels from each edge before scoring."""
    model.eval()
    crop = lambda t: t[..., border:t.shape[-2] - border, border:t.shape[-1] - border]
    scores, baselines, count = 0.0, 0.0, 0
    for lr, hr in loader:
        lr, hr = lr.to(DEVICE), hr.to(DEVICE)
        n = lr.size(0)
        scores += psnr(crop(model(lr)), crop(hr)) * n
        baselines += psnr(crop(bicubic(lr)), crop(hr)) * n
        count += n
    return scores / count, baselines / count


start_model, start_baseline = evaluate(model, val_loader)
# Wider than the model reaches: halo 4 LR pixels is 8 HR, and the skip adds 2 more.
inner_model, inner_baseline = evaluate(model, val_loader, border=2 * SKIP)

print(f"before training   whole patch   model {start_model:5.2f} dB   bicubic {start_baseline:5.2f} dB")
print(f"                  interior      model {inner_model:5.2f} dB   bicubic {inner_baseline:5.2f} dB")
assert abs(inner_model - inner_baseline) < 1e-3, "the model does not start at the baseline"
print("\nidentical in the interior, because the tail starts at zero and the skip *is* bicubic;")
print("the whole-patch gap is the zero-padded border ring, which tiling never shows")

In [ ]:
"""Train, measuring the held-out split after every epoch and keeping the best weights."""

optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)

history = {"loss": [], "psnr": [], "baseline": []}
best = {"psnr": float("-inf"), "epoch": -1, "state": None}
started = time.time()

for epoch in range(EPOCHS):
    model.train()
    running, seen = 0.0, 0

    for lr, hr in train_loader:
        lr, hr = lr.to(DEVICE), hr.to(DEVICE)
        loss = F.l1_loss(model(lr), hr)

        optimiser.zero_grad(set_to_none=True)
        loss.backward()
        optimiser.step()

        running += loss.item() * lr.size(0)
        seen += lr.size(0)

    schedule.step()
    val_psnr, val_baseline = evaluate(model, val_loader)
    history["loss"].append(running / seen)
    history["psnr"].append(val_psnr)
    history["baseline"].append(val_baseline)

    if val_psnr > best["psnr"]:
        best = {"psnr": val_psnr, "epoch": epoch,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}

    if epoch < 3 or (epoch + 1) % 10 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch + 1:3}/{EPOCHS}  l1 {running / seen:.4f}  "
              f"val {val_psnr:.2f} dB  bicubic {val_baseline:.2f} dB  "
              f"gain {val_psnr - val_baseline:+.2f} dB")

model.load_state_dict(best["state"])
print(f"\nbest epoch {best['epoch'] + 1} at {best['psnr']:.2f} dB, restored "
      f"({time.time() - started:.0f}s total)")

In [ ]:
"""Loss and PSNR against the baseline."""

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.5))
epochs = range(1, len(history["loss"]) + 1)

left.plot(epochs, history["loss"])
left.set_title("training L1 loss")
left.set_xlabel("epoch")

right.plot(epochs, history["psnr"], label="model")
right.plot(epochs, history["baseline"], "--", label="bicubic")
right.axvline(best["epoch"] + 1, color="0.8", zorder=0)
right.axvline(12, color="0.9", zorder=0)
right.annotate("12 epochs\n(upscale_nn)", (12, history["psnr"][11]), fontsize=8,
               xytext=(18, history["psnr"][11] - 2), arrowprops={"arrowstyle": "->"})
right.set_title("validation PSNR (dB)")
right.set_xlabel("epoch")
right.legend()

fig.tight_layout()
plt.show()

In [ ]:
"""What training bought, on the split it never trained on."""

final_psnr, final_baseline = evaluate(model, val_loader)
print(f"{'':>22}{'model':>9}{'bicubic':>10}{'gain':>11}")
for name, score in [("before training", start_model), (f"best epoch {best['epoch'] + 1}", final_psnr)]:
    print(f"{name:>22}{score:9.2f}{final_baseline:10.2f}{score - final_baseline:+10.2f} dB")
print(f"\n{len(val_set)} patches from {len(val_set.frames)} held-out frames")

In [ ]:
"""Nearest / bicubic / model / target, on validation patches."""

@torch.no_grad()
def compare(model, dataset, count=4):
    model.eval()
    picks = np.random.default_rng(SEED).choice(len(dataset), min(count, len(dataset)),
                                               replace=False)
    lr = torch.stack([dataset[int(i)][0] for i in picks]).to(DEVICE)
    hr = torch.stack([dataset[int(i)][1] for i in picks]).to(DEVICE)

    columns = {
        "nearest": F.interpolate(lr, scale_factor=SCALE, mode="nearest"),
        "bicubic": bicubic(lr),
        "model": model(lr),
        "target": hr,
    }

    fig, axes = plt.subplots(len(picks), 4, figsize=(11, 2.8 * len(picks)))
    axes = np.atleast_2d(axes)
    for row in range(len(picks)):
        for column, (name, images) in enumerate(columns.items()):
            image = images[row].clamp(0, 1).cpu().permute(1, 2, 0).numpy()
            axes[row][column].imshow(image, interpolation="nearest")
            axes[row][column].axis("off")
            if row == 0:
                axes[row][column].set_title(name)
    fig.tight_layout()
    plt.show()


compare(model, val_set)

## 4. Checkpoint — save it, load it back, check it survived

Same contract as `upscale_nn.ipynb`: the file carries everything needed to rebuild the
architecture, and it is loaded straight back into a fresh model and checked three ways —
same tensors, same output on a probe, same PSNR as the number stored beside them.

The halo travels with it, because the export and the browser both need it and neither
should be re-deriving it from the shape.

In [ ]:
"""Save the weights next to everything needed to rebuild the model."""

checkpoint_path = CHECKPOINT_DIR / "upscale_web.pt"
torch.save({
    "architecture": "WebUpscaler",
    "state_dict": model.state_dict(),
    "scale": SCALE,
    "widths": list(WIDTHS),
    "stem": STEM,
    "skip": SKIP,
    "halo": model.halo,
    "epochs": EPOCHS,
    "best_epoch": best["epoch"] + 1,
    "val_psnr": best["psnr"],
    "val_psnr_bicubic": final_baseline,
    "degradation": manifest["degradation"],
    "trained": time.strftime("%Y-%m-%dT%H:%M:%S"),
}, checkpoint_path)

print(f"{checkpoint_path.resolve()}  ({checkpoint_path.stat().st_size / 1e6:.2f} MB)")

In [ ]:
"""Load it back into a fresh model, and check the two are the same model."""

def load_upscaler(path, device):
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    restored = WebUpscaler(tuple(checkpoint["widths"]), checkpoint["stem"],
                           checkpoint["skip"], checkpoint["scale"]).to(device)
    restored.load_state_dict(checkpoint["state_dict"])
    restored.eval()
    return restored, checkpoint


loaded, checkpoint = load_upscaler(checkpoint_path, DEVICE)
for key, value in checkpoint.items():
    if key != "state_dict":
        print(f"{key:<18} {value}")

saved_state, loaded_state = model.state_dict(), loaded.state_dict()
assert saved_state.keys() == loaded_state.keys(), "the checkpoint lost a tensor"
assert all(torch.equal(saved_state[key].cpu(), loaded_state[key].cpu())
           for key in saved_state), "a tensor came back changed"

model.eval()
probe = torch.rand(2, 3, manifest["patch_lr"], manifest["patch_lr"], device=DEVICE)
with torch.no_grad():
    difference = float((model(probe) - loaded(probe)).abs().max())
assert difference <= 1e-6, f"the loaded model computes something else ({difference:.2e})"

loaded_psnr, _ = evaluate(loaded, val_loader)
assert abs(loaded_psnr - checkpoint["val_psnr"]) < 0.01, "the stored PSNR is not this model's"

print(f"\n{len(saved_state)} tensors identical, outputs identical "
      f"(max abs difference {difference:.1e})")
print(f"validation PSNR {loaded_psnr:.2f} dB, checkpoint claims {checkpoint['val_psnr']:.2f} dB")

In [ ]:
"""One whole frame through the loaded model, against bicubic, with a ground truth."""

@torch.no_grad()
def upscale_image(model, image, device):
    x = torch.from_numpy(np.asarray(image.convert("RGB")).transpose(2, 0, 1).copy())
    x = x.float().div_(255.0).unsqueeze(0).to(device)
    y = model(x).clamp(0, 1).squeeze(0).mul(255.0).round().byte()
    return Image.fromarray(y.cpu().permute(1, 2, 0).numpy())


source = sorted(RAW_DIR.glob("*.png"))[0]
truth = Image.open(source).convert("RGB")
truth = truth.crop((0, 0, truth.width - truth.width % SCALE, truth.height - truth.height % SCALE))
small = truth.resize((truth.width // SCALE, truth.height // SCALE), Image.BICUBIC)

started = time.time()
restored = upscale_image(loaded, small, DEVICE)
elapsed = time.time() - started
restored.save(SAMPLE_DIR / "upscaled_web.png")

box = (0, 0, min(320, truth.width), min(200, truth.height))
panels = {"bicubic": small.resize(truth.size, Image.BICUBIC).crop(box),
          f"model ({checkpoint['val_psnr']:.2f} dB)": restored.crop(box),
          "target": truth.crop(box)}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, (name, image) in zip(axes, panels.items()):
    ax.imshow(image, interpolation="nearest")
    ax.set_title(name, fontsize=10)
    ax.axis("off")
fig.tight_layout()
plt.show()

as_array = lambda image: np.asarray(image, dtype=np.float32) / 255.0
mse = lambda a, b: float(np.mean((a - b) ** 2))
truth_array = as_array(truth)
print(f"{source.name}  {small.size} -> {restored.size} in {elapsed * 1000:.0f} ms on {DEVICE.type}")
print(f"full frame PSNR  bicubic "
      f"{10 * np.log10(1 / mse(as_array(small.resize(truth.size, Image.BICUBIC)), truth_array)):.2f} dB"
      f"   model {10 * np.log10(1 / mse(as_array(restored), truth_array)):.2f} dB")

## 5. Export, and the optimisations that only exist at the other end

Everything from here is about the graph rather than the weights. Four candidates are
published — the same model at three precisions, plus the big-tile geometry — and each one
is checked against torch before it is timed, because a fast graph that computes something
else is not a faster model.

### float16

Half the weights, half the bytes across the bus, and the same PSNR to two decimals. The
conversion is exact enough here because the model is shallow and its activations live in
[0, 1]: the residual is small, and nothing accumulates far enough to need the exponent
range. `keep_io_types=False` is deliberate — leaving the I/O float32 would wrap the graph
in `Cast` and keep the upload at four bytes a channel, which is half of what fp16 is for.

**It does not show up on a desktop CPU.** The ONNX Runtime CPU provider has no half
kernels for these ops, so it casts to float32 and runs the same arithmetic — the number
below will be fp32's, or slightly worse. The win is on a GPU with `shader-f16`, and the
browser is where it was measured: **0.260 ms → 0.190 ms**, a 27% cut for a PSNR difference
that does not reach the second decimal.

### int8, which does not pay

Static QDQ quantization, calibrated on real training patches, costs **0.10 dB** (30.28 →
30.18) and produces a file 43% the size. It is also, on the WebGPU provider, **130×
slower**: 26 ms per tile against 0.19, because the `QuantizeLinear`/`DequantizeLinear`
pairs it inserts are outside the op budget and take the whole graph off the fast path —
and a quantized session refuses graph capture outright, so the one trick that would claw
some back is unavailable too. On WASM it is *also* slower than float32 (10.4 ms against
9.9). It is published anyway, as the row that shows why.

The rule it illustrates is worth more than the model: **in a browser, the op set decides
the speed, and the arithmetic barely gets a say.** A cheaper number type reached through
operators the provider does not implement is not cheaper.

### The tile is a dispatch, and dispatches are not free

Tiling has a third cost beside the halo and the memory: every tile is a `session.run`, and
each of those is JavaScript issuing commands. At step 128 a 1080p frame is 40 of them.
Measured on this model, in the browser, GPU-resident with graph capture:

| step | ms / tile | tiles / 1080p frame | ms / frame |
| --- | --- | --- | --- |
| 64 | 0.155 | 144 | 21 |
| 128 | 0.202 | 40 | 8 |
| 192 | 0.311 | 18 | 5 |
| 256 | 0.452 | 12 | **5** |
| 320 | 0.657 | 8 | **4** |

(One pass over five exports of these weights. The published step-256 model measures 0.42
in the final run below, so the shape of the curve holds rather than the third digit.)

Per-tile cost grows far slower than tile area, which is the shape of a fixed per-call
overhead: at step 64 almost the whole frame is dispatch, and the model itself is barely in
the number. The halo agrees — at halo 4, a step-128 tile computes 13% more pixels than it
keeps and a step-256 tile only 6%.

So the geometry is published twice: **step 128**, which is what every other model on the
page is measured at and the only one comparable with them, and **step 256**, which is what
a client should actually run.

In [ ]:
"""Export the trained model, and check every published graph against torch."""

exported, _ = load_upscaler(checkpoint_path, torch.device("cpu"))
halo = checkpoint["halo"]
size = webexport.TILE_STEP + 2 * halo


def content_tile(edge):
    """A real validation patch, edge-padded out to the tile a graph is pinned to."""
    patch = np.load(DATA_DIR / manifest["splits"]["val"]["lr"])[0]
    return np.pad(patch.transpose(2, 0, 1)[None].astype(np.float32) / 255.0,
                  ((0, 0), (0, 0), (0, edge - patch.shape[0]), (0, edge - patch.shape[1])),
                  mode="edge")


def check_graph(path, dtype=np.float32, tolerance=2e-3, tile=None, content=False):
    """Max absolute difference between the graph and the torch model it came from.

    `content` feeds a real patch instead of uniform noise, which is not a softer test but
    the *right* one for a calibrated graph: int8 ranges come from real activations, and
    noise drives them out of the range they were fitted to. Same graph, same weights: 0.011
    apart on content, 0.098 on noise. A quantized model is only equivalent on the
    distribution it was quantized for, and that is worth knowing before it is shipped.
    """
    edge = tile or size
    options = ort.SessionOptions()
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])
    sample = content_tile(edge) if content else np.random.rand(1, 3, edge, edge).astype(np.float32)
    expected = exported(torch.from_numpy(sample)).detach().numpy()
    actual = session.run(None, {session.get_inputs()[0].name: sample.astype(dtype)})[0]
    difference = float(np.abs(expected - actual.astype(np.float32)).max())
    assert difference < tolerance, f"{path.name} computes something else: {difference:.2e}"
    return difference


label = f"web {'-'.join(str(w) for w in WIDTHS)}"
fp32 = webexport.export(exported, f"upscale_web_tile{webexport.TILE_STEP}.onnx", size,
                        label=f"{label} fp32", halo=halo)
print(f"fp32       {check_graph(fp32['path']):.2e} vs torch   {fp32['kb']:6.1f} KB   "
      f"{', '.join(f'{op}x{n}' for op, n in sorted(fp32['ops'].items()))}")

In [ ]:
"""The same graph in half precision, and in int8 calibrated on real patches."""

fp16 = webexport.to_float16(fp32, f"upscale_web_fp16_tile{webexport.TILE_STEP}.onnx")
webexport.write_metadata(fp16["path"], {"label": f"{label} fp16"})
# float16 has ~3 decimal digits, so the graph is allowed to differ in the third.
print(f"fp16       {check_graph(fp16['path'], np.float16, 4e-3):.2e} vs torch   "
      f"{fp16['kb']:6.1f} KB")

# Calibration: real LR patches, padded out to the tile the graph is pinned to, so the
# activation ranges come from content rather than from noise.
train_lr = np.load(DATA_DIR / manifest["splits"]["train"]["lr"])
tiles = [np.pad(patch.transpose(2, 0, 1)[None].astype(np.float32) / 255.0,
                ((0, 0), (0, 0), (0, size - patch.shape[0]), (0, size - patch.shape[1])),
                mode="edge") for patch in train_lr[:48]]

int8 = webexport.to_int8(fp32, f"upscale_web_int8_tile{webexport.TILE_STEP}.onnx", tiles)
webexport.write_metadata(int8["path"], {"label": f"{label} int8 (Q/DQ, outside the budget)"})
print(f"int8       {check_graph(int8['path'], np.float32, 2e-2, content=True):.2e} vs torch "
      f"(on content)  {int8['kb']:6.1f} KB   "
      f"outside the budget: {', '.join(int8['outside_budget'])}")

# And the geometry a client should run: same weights, bigger step, fewer dispatches.
deploy_size = DEPLOY_STEP + 2 * halo
deploy = webexport.export(exported, f"upscale_webbig_tile{DEPLOY_STEP}.onnx", deploy_size,
                          label=f"{label} fp16, step {DEPLOY_STEP}", halo=halo,
                          step=DEPLOY_STEP)
deploy = webexport.to_float16(deploy, f"upscale_webbig_tile{DEPLOY_STEP}.onnx")
webexport.write_metadata(deploy["path"], {"label": f"{label} fp16, step {DEPLOY_STEP}",
                                          "step": DEPLOY_STEP, "halo": halo})
print(f"fp16 x{DEPLOY_STEP}  {check_graph(deploy['path'], np.float16, 4e-3, deploy_size):.2e} "
      f"vs torch   {deploy['kb']:6.1f} KB")

In [ ]:
"""PSNR of each published precision, measured through the runtime that will run it."""

val = manifest["splits"]["val"]
lr_patches, hr_patches = np.load(DATA_DIR / val["lr"]), np.load(DATA_DIR / val["hr"])


def graph_psnr(info, dtype=np.float32):
    """Quantization is judged where it lands, so this runs the *graph*, patch by patch.

    The published files are pinned to the tile, and the patches are 64px, so the copy
    measured here is a dynamic-axis export of the same weights through the same
    conversion - the precision is what is under test, not the shape.
    """
    options = ort.SessionOptions()
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(str(info["path"]), options, providers=["CPUExecutionProvider"])
    name = session.get_inputs()[0].name

    scores = []
    for lr, hr in zip(lr_patches, hr_patches):
        x = (lr.transpose(2, 0, 1)[None].astype(np.float32) / 255.0).astype(dtype)
        y = np.clip(session.run(None, {name: x})[0].astype(np.float32)[0].transpose(1, 2, 0), 0, 1)
        scores.append(10 * np.log10(1.0 / max(float(np.mean((y - hr / 255.0) ** 2)), 1e-12)))
    return float(np.mean(scores))


evaluation = CHECKPOINT_DIR / "evaluation"
evaluation.mkdir(exist_ok=True)
flexible = webexport.export(exported, "web_dynamic.onnx", manifest["patch_lr"],
                            halo=halo, dynamic=True, models_dir=evaluation)
flexible16 = webexport.to_float16(flexible, "web_dynamic_fp16.onnx", models_dir=evaluation)
flexible8 = webexport.to_int8(flexible, "web_dynamic_int8.onnx",
                              [patch.transpose(2, 0, 1)[None].astype(np.float32) / 255.0
                               for patch in train_lr[:48]], models_dir=evaluation)

print(f"{'precision':<10}{'PSNR':>8}{'vs float32':>12}{'KB':>9}{'ms/tile (CPU runtime)':>24}")
published = [("float32", fp32, flexible, np.float32), ("float16", fp16, flexible16, np.float16),
             ("int8", int8, flexible8, np.float32)]
quality = {}
for name, info, flexible_info, dtype in published:
    quality[name] = graph_psnr(flexible_info, dtype)
    ms = time_graph(info["path"], size, dtype)
    webexport.annotate(info, cpu_ms=round(ms, 2))
    print(f"{name:<10}{quality[name]:7.2f} dB{quality[name] - quality['float32']:+11.2f}"
          f"{info['kb']:9.1f}{ms:20.2f}")

webexport.annotate(deploy, cpu_ms=round(time_graph(deploy["path"], deploy_size, np.float16), 2))
print(f"\nfloat32 vs the torch model: {abs(quality['float32'] - checkpoint['val_psnr']):.3f} dB apart")

### Zero computation: the tile that did not change

The last optimisation is not in the model at all, and it is the largest one available to a
*desktop* stream: most of a desktop frame is identical to the frame before it. A tile whose
input has not changed has an output that has not changed either, so the model does not have
to run — the previous result is still correct, and reusing it costs a comparison instead of
a convolution.

This is exact rather than approximate, which is what makes it worth doing: it is not
skipping work that would have mattered, it is skipping work whose answer is already in
memory. The cost of deciding is one hash of the tile's input pixels, and the frame-to-frame
comparison it feeds.

Below, on a synthesised desktop sequence — a moving window, a moving cursor, and a block of
video playing — because there is no real capture in this repo to measure. A real session
would sit further towards the cheap end than this: a typing pause changes nothing at all,
and a full-screen video changes everything.

In [ ]:
"""How many tiles change between frames of a desktop-like sequence, and what that saves."""

def desktop_sequence(count, size=(1920, 1080), rng=None):
    """Frames of a drawn desktop: static chrome, one window dragged, a cursor, a video."""
    rng = rng or np.random.default_rng(SEED)
    width, height = size
    background = Image.new("RGB", size, (32, 36, 44))
    draw = ImageDraw.Draw(background)
    for y in range(height):                                   # wallpaper gradient
        draw.line([(0, y), (width, y)], fill=(32 + y // 40, 36 + y // 40, 44 + y // 40))
    for index in range(3):                                    # windows that stay put
        x, y = 60 + index * 520, 80 + index * 90
        draw.rectangle([x, y, x + 460, y + 620], fill=(238, 238, 240), outline=(90, 110, 160), width=2)
        draw.rectangle([x, y, x + 460, y + 30], fill=(90, 110, 160))
        for row in range(y + 50, y + 600, 16):                # text-like rows
            draw.rectangle([x + 16, row, x + 16 + int(rng.integers(120, 420)), row + 5],
                           fill=(40, 40, 48))

    frames = []
    for index in range(count):
        frame = background.copy()
        drawing = ImageDraw.Draw(frame)

        # The dragged window: 6 px a frame, which is a slow, ordinary drag.
        x, y = 1180 + 6 * index, 420 + 3 * index
        drawing.rectangle([x, y, x + 520, y + 380], fill=(250, 250, 252),
                          outline=(160, 90, 90), width=2)
        drawing.rectangle([x, y, x + 520, y + 28], fill=(160, 90, 90))

        # A block of video, entirely new every frame.
        block = rng.integers(0, 255, (180, 320, 3), dtype=np.uint8)
        frame.paste(Image.fromarray(block).resize((640, 360), Image.NEAREST), (200, 620))

        # The cursor.
        cx, cy = 900 + 11 * index, 300 + 7 * index
        drawing.polygon([(cx, cy), (cx + 14, cy + 40), (cx + 22, cy + 22)], fill=(255, 255, 255))
        frames.append(frame)
    return frames


def changed_tiles(frames, step, scale=SCALE):
    """Fraction of LR tiles that differ from the frame before, averaged over the sequence."""
    low = [np.asarray(frame.resize((frame.width // scale, frame.height // scale), Image.BICUBIC))
           for frame in frames]
    rows, columns = -(-low[0].shape[0] // step), -(-low[0].shape[1] // step)

    fractions = []
    for previous, current in zip(low, low[1:]):
        tile = lambda a, r, c: a[r * step:(r + 1) * step, c * step:(c + 1) * step]
        changed = sum(1 for r in range(rows) for c in range(columns)
                      if not np.array_equal(tile(current, r, c), tile(previous, r, c)))
        fractions.append(changed / (rows * columns))
    return float(np.mean(fractions)), rows * columns


frames = desktop_sequence(24)
frames[0].save(SAMPLE_DIR / "desktop_frame.png")

# Measured in the browser on an Ampere card, GPU-resident with graph capture, on the two
# geometries this notebook publishes - see the table at the end.
BROWSER_MS = {128: 0.190, 256: 0.424}

print(f"{'step':>5}{'tiles/frame':>13}{'changed':>10}{'every tile':>13}{'changed only':>14}")
for step, ms in BROWSER_MS.items():
    fraction, tiles = changed_tiles(frames, step)
    print(f"{step:5}{tiles:13}{fraction:9.0%}{tiles * ms:11.1f} ms"
          f"{tiles * fraction * ms:12.1f} ms")

print("\nthe same frame budget, spent only where the picture moved")

In [ ]:
"""What was published, and what the page will show."""

for info in [fp32, fp16, int8, deploy]:
    warning = (f"   outside the budget: {', '.join(info['outside_budget'])}"
               if info["outside_budget"] else "")
    print(f"{info['path'].name:<34} {info['kb']:7.1f} KB{warning}")
print(f"\nwritten to {webexport.MODELS_DIR.resolve()}")

### What the browser said

Run the page and the numbers below are the ones to reproduce:

```
uv run upscale/benchmark/main.py       # from model/, then open http://127.0.0.1:8000
```

Every number below is a **median of three interleaved rounds** rather than one run: on
this card the same configuration drifts by up to 10% between passes, which is wider than
some of the differences being claimed, and running the configurations in turn rather than
in blocks is what keeps the drift out of the comparison. The models are the four files
this notebook just published.

**One tile, NVIDIA Ampere, Chromium, WebGPU, GPU-resident, graph capture:**

| model | ms / tile | 1080p frame | fps | val PSNR |
| --- | --- | --- | --- | --- |
| static bicubic (`Resize` only) | 0.038 | 2 ms | 664 | 23.41 dB |
| `upscale_nn` — 64-32 with `Resize` | 0.447 | 18 ms | 56 | — |
| web 32-16 fp32 | 0.260 | 10 ms | 96 | 30.28 dB |
| **web 32-16 fp16** | **0.190** | **7 ms** | **148** | 30.28 dB |
| web 32-16 fp16, step 256 | 0.424 | **5 ms** | 197 | 30.28 dB |
| web 32-16 int8 | 26.2 | 1049 ms | 1 | 30.18 dB |

**The runtime ladder**, same model (32-16 fp16), same card, four interleaved rounds — none
of this is a property of the model, all of it is how the page runs it:

| how the run is made | ms / tile |
| --- | --- |
| CPU round trip (upload the tile and download the result, every run) | 3.24 |
| GPU-resident, new output tensor per run | 0.300 |
| GPU-resident, output buffer reused | 0.250 |
| GPU-resident, reused, **graph capture** | **0.208** |

The round trip is the one to notice: **it is 16× the model and it is the same 3.2 ms for
every model on the page**, so a client that hands ONNX Runtime a JavaScript array is not
running a fast model slowly, it is not running the model at all — it is timing a copy. The
rest is allocation and dispatch: a fresh output buffer every run costs 17%, and the
JavaScript that issues the commands costs 17% again, which graph capture records once and
replays.

**WASM, no GPU:** 9.9 ms per tile for this model (396 ms a frame), against 44.9 ms for
`upscale_nn` and 20.5 ms for the static bicubic. Nothing runs at video rate without a GPU
— but note the ordering, because it is not the WebGPU one: `Resize` is the *fastest* model
on the page under WebGPU and among the slowest under WASM, fp16 is a hair slower than fp32
rather than 27% faster, and int8 loses under both. Which backend a number came from is
part of the number.